# Variant Scoring with the Nucleotide Transformer on a T4 GPU

**Workshop: AI on AnVIL — genomic language models for variant effect prediction**

This notebook demonstrates *zero-shot* variant effect scoring with a pretrained DNA
language model (the Nucleotide Transformer, InstaDeepAI). The idea:

- The model is trained only to predict masked nucleotides in genomic sequence — it has
  never seen a disease label.
- If we mask the exact position of a variant and ask the model "how likely is the
  reference base here vs. the alternate base?", a large gap (reference strongly
  preferred over alternate) suggests the alternate allele disrupts a sequence pattern
  the model learned to be important — a proxy for a functional/deleterious effect.
- No variant-specific training is required, which is what makes this a compelling
  illustration of an LLM-style genomic model versus a conventional supervised classifier.

**Steps:**
1. Check GPU availability (T4) and load a mid-size Nucleotide Transformer checkpoint.
2. Fetch a handful of real, well-known human variants (with reference sequence context
   and ClinVar-style significance) from the Ensembl REST API.
3. Score each variant with the model's masked-token log-likelihood ratio.
4. Compare scores against each variant's reported clinical significance.
5. See how to point this at a VCF from your own AnVIL workspace instead of the toy list.

> **Caveat for the workshop:** this is an illustrative, tiny-N demo, not a validated
> clinical classifier. Zero-shot genomic LM scores are a research tool, not a
> replacement for CADD, AlphaMissense, SpliceAI, REVEL, or ACMG-based clinical review.

## 1. Check the GPU

Confirm the runtime has a GPU (a T4 is plenty for the 500M-parameter model used below).
If this doesn't show a GPU, attach one to the notebook's runtime before continuing —
everything below will still run on CPU, just much more slowly.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q transformers accelerate torch requests matplotlib

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Load the Nucleotide Transformer

Using the 500M-parameter multi-species v2 checkpoint — small enough to load in fp16 on
a T4 (~1 GB of weights) while still being a real, capable genomic LM. `trust_remote_code`
is required because InstaDeepAI ships a custom tokenizer/model implementation.

If this checkpoint name has moved, browse current options at
https://huggingface.co/InstaDeepAI and swap in the model id below (e.g. the
`nucleotide-transformer-500m-human-ref` variant, trained only on the human reference,
is a reasonable alternative for a human-variant-only demo).

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
model.eval()

mask_token_id = tokenizer.mask_token_id
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME} ({n_params / 1e6:.0f}M params)")
print(f"Mask token: {tokenizer.mask_token!r} (id={mask_token_id})")

## 3. Pull real variants + reference context from Ensembl

Rather than hard-coding genomic coordinates (easy to get subtly wrong), this fetches
each variant live from the Ensembl REST API by rsID: its GRCh38 coordinates, its
ClinVar-derived clinical significance (when Ensembl has one on file), and a window of
flanking reference sequence. The reference base at the variant position is checked
against the expected allele as a sanity check, and strand is handled explicitly, since
`allele_string` can be reported on either strand while fetched sequence is always
plus-strand.

In [ ]:
import requests

ENSEMBL_REST = "https://rest.ensembl.org"
WINDOW = 100  # bases of context on each side of the variant
COMPLEMENT = str.maketrans("ACGT", "TGCA")


def fetch_variant_context(rsid, window=WINDOW):
    """Resolve an rsID to GRCh38 coordinates + flanking reference sequence via Ensembl.
    Returns a dict, or None if the variant can't be resolved as a simple biallelic SNV.
    """
    r = requests.get(
        f"{ENSEMBL_REST}/variation/human/{rsid}",
        params={"content-type": "application/json"},
        timeout=15,
    )
    if r.status_code != 200:
        print(f"[{rsid}] lookup failed: HTTP {r.status_code}")
        return None
    data = r.json()

    mappings = [
        m for m in data.get("mappings", [])
        if str(m.get("assembly_name", "")).startswith("GRCh38")
    ]
    if not mappings:
        print(f"[{rsid}] no GRCh38 mapping found")
        return None
    mapping = mappings[0]
    chrom = mapping["seq_region_name"]
    pos = mapping["start"]
    allele_string = mapping["allele_string"]

    if "/" not in allele_string or len(allele_string.split("/")) != 2:
        print(f"[{rsid}] not a simple biallelic variant ({allele_string}); skipping")
        return None
    ref, alt = allele_string.split("/")
    if len(ref) != 1 or len(alt) != 1:
        print(f"[{rsid}] not a single-nucleotide variant ({allele_string}); skipping")
        return None

    # allele_string is relative to the variant's submitted strand; fetched sequence
    # below is always plus-strand, so flip the alleles if needed.
    if mapping.get("strand", 1) == -1:
        ref = ref.translate(COMPLEMENT)
        alt = alt.translate(COMPLEMENT)

    seq_r = requests.get(
        f"{ENSEMBL_REST}/sequence/region/human/{chrom}:{pos - window}-{pos + window}",
        params={"content-type": "text/plain"},
        timeout=15,
    )
    if seq_r.status_code != 200:
        print(f"[{rsid}] sequence fetch failed: HTTP {seq_r.status_code}")
        return None
    flank = seq_r.text.strip().upper()

    center = window  # index of the variant base within `flank`
    if flank[center] != ref.upper():
        print(f"[{rsid}] reference base mismatch: expected {ref}, got {flank[center]} - skipping")
        return None

    return {
        "rsid": rsid,
        "chrom": chrom,
        "pos": pos,
        "ref": ref.upper(),
        "alt": alt.upper(),
        "clinical_significance": data.get("clinical_significance", []),
        "ref_sequence": flank,
        "alt_sequence": flank[:center] + alt.upper() + flank[center + 1:],
    }

## 4. Example variant set

A handful of well-known, real human SNPs — deliberately resolved live via the API
above rather than assumed, so coordinates and significance labels are always current.
Swap these rsIDs for anything relevant to your audience.

In [ ]:
EXAMPLE_RSIDS = [
    "rs334",       # HBB - sickle cell allele
    "rs6025",      # F5 - Factor V Leiden
    "rs7412",      # APOE
    "rs429358",    # APOE
    "rs1800562",   # HFE - hemochromatosis (C282Y)
    "rs1801133",   # MTHFR - C677T
    "rs1799990",   # PRNP - M129V
]

variants = []
for rsid in EXAMPLE_RSIDS:
    v = fetch_variant_context(rsid)
    if v is not None:
        variants.append(v)

print(f"Resolved {len(variants)}/{len(EXAMPLE_RSIDS)} variants")
variants

## 5. Zero-shot scoring

Tokenize the reference and alternate sequences independently, find the single token
that differs between them, mask that position in the reference sequence, and read off
the model's log-probability for the reference token vs. the alternate token at that
masked position. The score is `log P(alt) - log P(ref)`: more negative means the model
finds the alternate allele much less likely than the reference in that context.

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def score_variant(ref_sequence, alt_sequence):
    ref_ids = tokenizer(ref_sequence, return_tensors="pt")["input_ids"][0]
    alt_ids = tokenizer(alt_sequence, return_tensors="pt")["input_ids"][0]

    if ref_ids.shape != alt_ids.shape:
        raise ValueError("ref/alt tokenized to different lengths - unexpected for an SNV")

    diff_positions = (ref_ids != alt_ids).nonzero(as_tuple=True)[0]
    if len(diff_positions) != 1:
        raise ValueError(f"expected exactly one differing token, found {len(diff_positions)}")
    idx = diff_positions[0].item()

    ref_token_id = ref_ids[idx].item()
    alt_token_id = alt_ids[idx].item()

    masked_ids = ref_ids.clone()
    masked_ids[idx] = mask_token_id
    input_ids = masked_ids.unsqueeze(0).to(device)

    logits = model(input_ids=input_ids).logits[0, idx]
    log_probs = F.log_softmax(logits.float(), dim=-1)

    return (log_probs[alt_token_id] - log_probs[ref_token_id]).item()

In [ ]:
results = []
for v in variants:
    try:
        s = score_variant(v["ref_sequence"], v["alt_sequence"])
    except Exception as e:
        print(f"[{v['rsid']}] scoring failed: {e}")
        continue
    sig = ", ".join(v["clinical_significance"]) if v["clinical_significance"] else "(none reported)"
    results.append({**v, "score": s, "clinsig_label": sig})
    print(f"{v['rsid']:<12} {v['chrom']}:{v['pos']:<12} {v['ref']}>{v['alt']}  "
          f"score={s:+.3f}   ClinVar: {sig}")

## 6. Plot: do scores separate pathogenic from benign?

In [ ]:
import matplotlib.pyplot as plt

labels = [r["rsid"] for r in results]
scores = [r["score"] for r in results]
sigs = [r["clinsig_label"] for r in results]
colors = ["#c0392b" if "pathogenic" in s.lower() else "#2980b9" for s in sigs]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, scores, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("log P(alt) - log P(ref)")
ax.set_title("Zero-shot Nucleotide Transformer variant scores")
plt.xticks(rotation=30, ha="right")
for bar, sig in zip(bars, sigs):
    ax.annotate(sig, (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                textcoords="offset points", xytext=(0, 4), ha="center",
                fontsize=7, rotation=90)
plt.tight_layout()
plt.show()

## 7. Discussion points for the workshop

- **Tiny N, not a benchmark.** Seven variants is enough to show the mechanics, not to
  claim the model "works." A real evaluation would score thousands of ClinVar
  pathogenic/benign SNVs and report AUROC, as the Nucleotide Transformer, GPN, and
  related papers do.
- **`clinical_significance` comes straight from Ensembl's ClinVar mirror** and can be
  sparse, outdated, or conflicting for some IDs — worth a live caveat if a bar shows
  "(none reported)".
- **Strand handling matters.** The mismatch check and strand-flip in
  `fetch_variant_context` are there because skipping this step silently produces wrong
  reference alleles for minus-strand submissions — a good moment to flag common
  correctness pitfalls in genomics pipelines generally.
- **This is a proxy, not a diagnosis.** A low score means "the model finds this allele
  surprising in its sequence context," which correlates with functional importance but
  is not equivalent to clinical pathogenicity.

## 8. Scaling this to your own AnVIL workspace

Swap the toy `EXAMPLE_RSIDS` list for variants pulled directly from a workspace VCF,
and read reference sequence locally instead of one Ensembl call per variant (much
faster for a full cohort):

```python
# from cyvcf2 import VCF
#
# vcf_path = "gs://fc-xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx/path/to/cohort.vcf.gz"
# workspace_variants = []
# for rec in VCF(vcf_path):
#     if len(rec.REF) == 1 and len(rec.ALT[0]) == 1:
#         workspace_variants.append({
#             "chrom": rec.CHROM.replace("chr", ""),
#             "pos": rec.POS,
#             "ref": rec.REF,
#             "alt": rec.ALT[0],
#         })
#
# Then replace the Ensembl sequence lookup with a local reference FASTA read
# (e.g. pyfaidx against the workspace's reference genome) to avoid one HTTP
# request per variant when scoring an entire cohort.
```

**GPU note:** the 500M model comfortably fits a T4 in fp16 with room to batch dozens of
variants at once (stack multiple masked sequences into one forward pass) if you want to
scale this beyond a one-at-a-time loop.